In [ ]:
!pip install cfgrib

In [ ]:
import os, gc, shutil, pickle, warnings
import sys
import numpy as np
import xarray as xr
import cfgrib
import pandas as pd
from scipy.stats import pearsonr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings("ignore")
os.environ['PYTHONUNBUFFERED'] = '1'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def log(msg):
    print(msg, flush=True)
    sys.stdout.flush()


gribpaths = [
    "/kaggle/input/datasets/oussemabouhajeb/weather-2023-full/33761554a62d5e819b138aee10277313.grib",
    "/kaggle/input/datasets/oussemabouhajeb/weather2024/ee84a1d956d86ffbe87ac013c7e3eaa5.grib",
    "/kaggle/input/datasets/oussemabouhajeb/weather-full-2022/1e9482f8888a5f6408ab063706a223d3.grib",
    "/kaggle/input/datasets/oussemabouhajeb/weather-2019-full/85bfe7e0b20dcd530bf6e730ecd3e83e.grib",
]

keepvars = ['u10', 'v10', 't2m', 'msl', 'sst', 'swh']

yearlydatasets = []

for path in gribpaths:
    if not os.path.exists(path):
        continue
    try:
        tmppath = f"/tmp/{os.path.basename(path)}"
        for f in os.listdir("/tmp"):
            if f.endswith(".idx"):
                os.remove(f"/tmp/{f}")
        shutil.copy2(path, tmppath)

        datasets = cfgrib.open_datasets(tmppath)
        groupdatasets = []
        reflat = reflon = None

        for idx, d in enumerate(datasets):
            try:
                if "valid_time" in d.coords and "time" not in d.dims:
                    d = d.rename({"valid_time": "time"})
                coordstodrop = [c for c in ['step','number','surface','heightAboveGround','meanSea','depthBelowLandLayer']
                                if c in d.coords or c in d.dims]
                d = d.drop_vars(coordstodrop, errors='ignore')
                existing = [v for v in keepvars if v in d]
                if not existing:
                    continue
                d = d[existing]
                if "latitude" in d.dims:
                    lats = d.latitude.values
                    if lats[0] > lats[-1]:
                        d = d.sel(latitude=slice(46, 30), longitude=slice(-6, 37))
                    else:
                        d = d.sel(latitude=slice(30, 46), longitude=slice(-6, 37))

                d = d.sortby('time')

                log(f"  [{os.path.basename(path)}] sub-ds {idx}: "
                      f"freq={pd.infer_freq(pd.DatetimeIndex(d.time.values[:10]))}, "
                      f"steps={len(d.time)}, "
                      f"first={d.time.values[0]}")

                if reflat is None:
                    reflat = d.latitude
                    reflon = d.longitude
                elif len(d.latitude) != len(reflat):
                    d = d.interp(latitude=reflat, longitude=reflon)
                groupdatasets.append(d)
            except Exception as e:
                log(f"  sub-ds {idx} failed: {e}")
                pass

        if not groupdatasets:
            continue

        if len(groupdatasets) == 1:
            dsyear = groupdatasets[0]
        else:
            tstart = max(g.time.values[0]  for g in groupdatasets)
            tend   = min(g.time.values[-1] for g in groupdatasets)
            aligned = [g.sel(time=slice(tstart, tend)) for g in groupdatasets]
            dsyear = xr.merge(aligned, compat="override", join="inner")

        yearlydatasets.append(dsyear)
        del datasets, dsyear, groupdatasets
        gc.collect()

    except Exception as e:
        log(f"Failed to load {path}: {e}")
        pass

if yearlydatasets:
    dsweather = xr.concat(yearlydatasets, dim="time").sortby("time")
    _, uniq = np.unique(dsweather.time.values, return_index=True)
    dsweather = dsweather.isel(time=uniq)
    del yearlydatasets; gc.collect()
else:
    raise RuntimeError("No datasets loaded!")

for path in gribpaths:
    tmppath = f"/tmp/{os.path.basename(path)}"
    try: os.remove(tmppath)
    except: pass


log(f"\nFinal dataset: {len(dsweather.time)} hourly timesteps")
log(f"Time range: {dsweather.time.values[0]}  →  {dsweather.time.values[-1]}")


# ── Feature Engineering ────────────────────────────────────────────────────────
def createfeatures(ds):
    ds['windspeed'] = np.sqrt(ds['u10']**2 + ds['v10']**2)
    if 'mwd' in ds:
        ds['mwdsin'] = np.sin(np.radians(ds['mwd']))
        ds['mwdcos'] = np.cos(np.radians(ds['mwd']))
    if 'msl' in ds:
        ds['deltamsl'] = ds['msl'].diff(dim='time').fillna(0)
    if 'd2m' in ds and 't2m' in ds:
        ds['dewpointdep'] = ds['t2m'] - ds['d2m']

    dayofyear = ds.time.dt.dayofyear
    hourofday = ds.time.dt.hour
    doysin  = np.sin(2 * np.pi * dayofyear / 365).values
    doycos  = np.cos(2 * np.pi * dayofyear / 365).values
    hodsin  = np.sin(2 * np.pi * hourofday / 24).values
    hodcos  = np.cos(2 * np.pi * hourofday / 24).values

    latlen, lonlen = len(ds.latitude), len(ds.longitude)
    for name, arr in [('doysin', doysin), ('doycos', doycos),
                      ('hodsin', hodsin), ('hodcos', hodcos)]:
        ds[name] = xr.DataArray(
            arr[:, None, None] * np.ones((1, latlen, lonlen)),
            dims=['time', 'latitude', 'longitude'],
            coords={'time': ds.time, 'latitude': ds.latitude, 'longitude': ds.longitude}
        )
    return ds

dsweather = createfeatures(dsweather.sortby('latitude'))

# ── Land Masks ─────────────────────────────────────────────────────────────────
# SST mask: standard 0.25° ocean boundary
sst_landmask = np.isnan(dsweather['sst'].isel(time=0).values).astype(np.float32)

# SWH mask: SWH is originally 0.5°, interpolated to 0.25°.
# After interp, land NaNs get filled → use time-mean NaN + union with SST mask
# to recover land points that interpolation silently filled.
swh_nanmask  = np.isnan(dsweather['swh'].mean(dim='time').values).astype(np.float32)
swh_landmask = np.clip(sst_landmask + swh_nanmask, 0, 1).astype(np.float32)

# Default mask for all other variables (atm variables are all 0.25°)
landmask = sst_landmask

log(f"SST land points : {int(sst_landmask.sum())}")
log(f"SWH land points : {int(swh_landmask.sum())}  (should be >= SST)")

# Map each forecast variable to its correct land mask
VAR_MASKS = {
    'swh'      : swh_landmask,
    'mwp'      : swh_landmask,   # same 0.5° origin as swh
    't2m'      : sst_landmask,
    'msl'      : sst_landmask,
    'sst'      : sst_landmask,
    'u10'      : sst_landmask,
    'v10'      : sst_landmask,
    'windspeed': sst_landmask,
}

if 'msl' in dsweather:
    dsweather['msl'] = dsweather['msl'] / 100.0
if 'sp' in dsweather:
    dsweather['sp'] = dsweather['sp'] / 100.0

atmvars = ['u10','v10','t2m','msl','d2m','sp','tp','windspeed','deltamsl','dewpointdep',
           'doysin','doycos','hodsin','hodcos']
for var in atmvars:
    if var in dsweather:
        dsweather[var] = dsweather[var].fillna(0)


# ── Train/Val/Test Split ───────────────────────────────────────────────────────
trainratio = 0.70
splitindex = int(len(dsweather.time) * trainratio)

varstonormalize = ['u10','v10','t2m','msl','sp','sst','swh','mwp','windspeed','dewpointdep','deltamsl']

normstats = {}

for var in varstonormalize:
    if var not in dsweather:
        continue
    arr = dsweather[var].values.astype(np.float32)
    trainarr = arr[:splitindex]
    mean = float(np.nanmean(trainarr))
    std  = float(np.nanstd(trainarr))
    if std < 1e-8:
        normstats[var] = {'mean': mean, 'std': 1.0}
        continue
    arrnorm = (arr - mean) / std
    dsweather[var] = xr.DataArray(arrnorm, dims=dsweather[var].dims,
                                   coords=dsweather[var].coords, attrs=dsweather[var].attrs)
    normstats[var] = {'mean': mean, 'std': std}

with open("normstats.pkl", "wb") as f:
    pickle.dump(normstats, f)


forecastvars = [v for v in ['swh','u10','v10','t2m','msl','sst','windspeed'] if v in dsweather]

allarrays = {var: dsweather[var].values.astype(np.float32) for var in forecastvars}
allarrays = {k: np.nan_to_num(v, nan=0.0) for k, v in allarrays.items()}

# ── Hyperparameters ────────────────────────────────────────────────────────────
seqlen    = 48
forecasth = [5]

T, H, W = next(iter(allarrays.values())).shape
log(f"Array shape: T={T}, H={H}, W={W}")

del dsweather; gc.collect()


# ── Datasets ───────────────────────────────────────────────────────────────────
class SingleVarDataset(Dataset):
    def __init__(self, data, seqlen, horizons, startidx, endidx, stride=6):
        self.data = data
        self.seqlen = seqlen
        self.horizons = horizons
        maxh = max(horizons)
        self.indices = [i for i in range(startidx, endidx, stride)
                        if i - seqlen >= 0 and i + maxh < len(data)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        t  = self.indices[k]
        x  = self.data[t - self.seqlen : t]
        yt = self.data[t]
        yf = np.stack([self.data[t + h] for h in self.horizons], axis=0)
        return (
            torch.from_numpy(x).unsqueeze(1),
            torch.from_numpy(yt),
            torch.from_numpy(yf),
        )


# ── Model Definitions ──────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, inch, hiddench, kernel=3):
        super().__init__()
        pad = kernel // 2
        self.hiddench = hiddench
        self.gates = nn.Conv2d(inch + hiddench, 4 * hiddench, kernel, padding=pad)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates    = self.gates(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i); f = torch.sigmoid(f)
        g = torch.tanh(g);    o = torch.sigmoid(o)
        cnew = f * c + i * g
        hnew = o * torch.tanh(cnew)
        return hnew, cnew


class ConvLSTMEncoder(nn.Module):
    def __init__(self, inch=1, hiddench=32):
        super().__init__()
        self.cell1    = ConvLSTMCell(inch,     hiddench)
        self.cell2    = ConvLSTMCell(hiddench, hiddench)
        self.hiddench = hiddench
        self.bn       = nn.BatchNorm2d(hiddench)

    def forward(self, x):
        B, T, C, Hdim, Wdim = x.shape
        h1 = torch.zeros(B, self.hiddench, Hdim, Wdim, device=x.device)
        c1 = torch.zeros_like(h1)
        h2 = torch.zeros_like(h1)
        c2 = torch.zeros_like(h1)
        for t in range(T):
            h1, c1 = self.cell1(x[:, t], h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)
        return self.bn(h2)


class ReconstructionBranch(nn.Module):
    def __init__(self, hiddench=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(hiddench, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1))

    def forward(self, z):
        return self.net(z).squeeze(1)


class ForecastingBranch(nn.Module):
    def __init__(self, hiddench=32):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(hiddench, hiddench, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hiddench, hiddench, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hiddench, hiddench // 2, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hiddench // 2, 1, 1)
        )

    def forward(self, z):
        return self.head(z).squeeze(1).unsqueeze(1)


class VariableModel(nn.Module):
    def __init__(self, hiddench=32):
        super().__init__()
        self.encoder  = ConvLSTMEncoder(inch=1, hiddench=hiddench)
        self.recon    = ReconstructionBranch(hiddench)
        self.forecast = ForecastingBranch(hiddench)

    def forward(self, x):
        z      = self.encoder(x)
        r      = self.recon(z)
        deltas = self.forecast(z)
        last   = x[:, -1, 0]
        f      = last.unsqueeze(1) + deltas
        return r, f


# ── Hyperparameters ────────────────────────────────────────────────────────────
lambda1      = 0.1
lambda2      = 0.9
batchsize    = 32
maxepochs    = 20
patience     = 4
lr           = 1e-3
hiddench     = 32

landmasktensor     = torch.from_numpy(landmask).to(device)
swh_landmasktensor = torch.from_numpy(swh_landmask).to(device)

# Map each variable to its mask tensor (used in training)
VAR_MASK_TENSORS = {
    'swh'      : swh_landmasktensor,
    'mwp'      : swh_landmasktensor,
    't2m'      : landmasktensor,
    'msl'      : landmasktensor,
    'sst'      : landmasktensor,
    'u10'      : landmasktensor,
    'v10'      : landmasktensor,
    'windspeed': landmasktensor,
}


# ── Loss Functions ─────────────────────────────────────────────────────────────
def maskedmse(pred, target, mask):
    ocean = 1.0 - mask
    loss  = ((pred - target) ** 2) * ocean
    return loss.sum() / (ocean.sum() * pred.shape[0]).clamp(min=1)


def maskedmae(pred, target, mask):
    ocean = 1.0 - mask
    loss  = torch.abs(pred - target) * ocean
    return loss.sum() / (ocean.sum() * pred.shape[0]).clamp(min=1)


# ── Training Functions ─────────────────────────────────────────────────────────
def trainvariable(varname, dataarr):
    # ── use the correct mask for this variable
    varmask = VAR_MASK_TENSORS.get(varname, landmasktensor)

    ntrain = int(T * trainratio)
    nval   = int(T * 0.15)

    trainds = SingleVarDataset(dataarr, seqlen, forecasth, seqlen,        ntrain)
    valds   = SingleVarDataset(dataarr, seqlen, forecasth, ntrain,        ntrain + nval)
    testds  = SingleVarDataset(dataarr, seqlen, forecasth, ntrain + nval, T)

    trainloader = DataLoader(trainds, batch_size=batchsize, shuffle=True,  num_workers=2, pin_memory=True)
    valloader   = DataLoader(valds,   batch_size=batchsize, shuffle=False, num_workers=2, pin_memory=True)
    testloader  = DataLoader(testds,  batch_size=batchsize, shuffle=False, num_workers=2, pin_memory=True)

    model     = VariableModel(hiddench=hiddench).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    bestval = float('inf'); patiencectr = 0
    trainlosses, vallosses = [], []

    for epoch in range(1, maxepochs + 1):
        log(f"  [{varname}] Starting epoch {epoch}...")
        model.train(); epochloss = 0.0
        for x, yt, yf in trainloader:
            x, yt, yf = x.to(device), yt.to(device), yf.to(device)
            optimizer.zero_grad()
            r, f  = model(x)
            lossr = maskedmse(r, yt, varmask)
            lossf = maskedmse(f[:, 0], yf[:, 0], varmask)
            loss  = lambda1 * lossr + lambda2 * lossf
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            epochloss += loss.item()

        epochloss /= len(trainloader); trainlosses.append(epochloss)

        model.eval(); valloss = 0.0
        with torch.no_grad():
            for x, yt, yf in valloader:
                x, yt, yf = x.to(device), yt.to(device), yf.to(device)
                r, f  = model(x)
                lossr = maskedmse(r, yt, varmask)
                lossf = maskedmse(f[:, 0], yf[:, 0], varmask)
                valloss += (lambda1 * lossr + lambda2 * lossf).item()
        valloss /= len(valloader); vallosses.append(valloss)
        scheduler.step(valloss)

        log(f"  [{varname}] Epoch {epoch:3d}/{maxepochs} | Train={epochloss:.4f} | Val={valloss:.4f}")

        if valloss < bestval:
            bestval = valloss; patiencectr = 0
            torch.save(model.state_dict(), f"best_{varname}.pt")
        else:
            patiencectr += 1
            if patiencectr >= patience:
                log(f"  [{varname}] Early stopping at epoch {epoch}"); break

    model.load_state_dict(torch.load(f"best_{varname}.pt"))
    return model, testloader, trainlosses, vallosses


# ── Train All Models ───────────────────────────────────────────────────────────
windvars    = {'windspeed'}
nonwindvars = [v for v in forecastvars if v not in windvars]

allmodels    = {}
allresults   = {}
allhistories = {}

for var in nonwindvars:
    log(f"\n{'='*40}")
    log(f"Starting training: {var}")
    log(f"{'='*40}")
    model, testloader, trainhist, valhist = trainvariable(var, allarrays[var])
    allmodels[var]    = model
    allhistories[var] = {'train': trainhist, 'val': valhist}
    log(f"✓ Training done: {var}")
    torch.cuda.empty_cache(); gc.collect()


# ── Evaluation ─────────────────────────────────────────────────────────────────
with open("normstats.pkl", "rb") as f:
    normstats = pickle.load(f)

def denorm(arr, var):
    if var not in normstats:
        return arr
    return arr * normstats[var]['std'] + normstats[var]['mean']

# ── use variable-specific ocean masks for evaluation too
oceanmasks = {var: (1 - VAR_MASKS[var]).astype(bool) for var in VAR_MASKS}

units = {
    'swh': 'm', 't2m': 'K', 'msl': 'hPa',
    'sst': 'K', 'u10': 'm/s', 'v10': 'm/s', 'windspeed': 'm/s'
}

cpu = torch.device('cpu')

def collectpredsreal(model, testloader, varname):
    model.eval()
    model.to(cpu)
    omask = oceanmasks.get(varname, (1 - landmask).astype(bool))
    allpreds   = []
    alltargets = []
    with torch.no_grad():
        for x, yt, yf in testloader:
            _, f = model(x)
            f   = f.numpy()
            yf  = yf.numpy()
            for b in range(f.shape[0]):
                allpreds.append(  denorm(f[b, 0][omask],  varname))
                alltargets.append(denorm(yf[b, 0][omask], varname))
    return np.concatenate(allpreds), np.concatenate(alltargets)


def computemetrics(pred, target):
    rmse    = np.sqrt(np.mean((pred - target) ** 2))
    mae     = np.mean(np.abs(pred - target))
    corr, _ = pearsonr(pred, target)
    thr05   = 0.5 * rmse
    thr10   = 1.0 * rmse
    thr20   = 2.0 * rmse
    acc05   = 100 * np.mean(np.abs(pred - target) <= thr05)
    acc10   = 100 * np.mean(np.abs(pred - target) <= thr10)
    acc20   = 100 * np.mean(np.abs(pred - target) <= thr20)
    return {'rmse': rmse, 'mae': mae, 'corr': corr,
            'acc05': acc05, 'acc10': acc10, 'acc20': acc20}


def printvarresults(varname, pred, target):
    unit = units.get(varname, '?')
    m    = computemetrics(pred, target)
    log(f"\n{'='*60}")
    log(f"  {varname.upper()}  [{unit}]   (horizon = t+1h)")
    log(f"  Tolerances  strict: ±{0.5*m['rmse']:.4f}  |  fair: ±{m['rmse']:.4f}  |  loose: ±{2*m['rmse']:.4f}  [{unit}]")
    log(f"{'='*60}")
    log(f"  {'rmse':>8} {'mae':>8} {'corr':>7} {'strict%':>9} {'fair%':>8} {'loose%':>8}")
    log(f"  {m['rmse']:>8.4f} {m['mae']:>8.4f} {m['corr']:>7.4f} "
          f"{m['acc05']:>8.2f}% {m['acc10']:>7.2f}% {m['acc20']:>7.2f}%")
    return m


finalresults = {}
ntrain = int(T * trainratio)
nval   = int(T * 0.15)

trained_nonwind = [var for var in nonwindvars if var in allmodels]

for var in trained_nonwind:
    log(f"\nEvaluating {var}...")
    testds     = SingleVarDataset(allarrays[var], seqlen, forecasth, ntrain + nval, T)
    testloader = DataLoader(testds, batch_size=batchsize, shuffle=False, num_workers=2, pin_memory=False)
    pred, target = collectpredsreal(allmodels[var], testloader, var)
    m = printvarresults(var, pred, target)
    finalresults[var] = m


# ── Summary Table ──────────────────────────────────────────────────────────────
log(f"\n{'='*80}")
log(f"  SUMMARY  —  t+1h forecast  |  all values in native units after denorm")
log(f"{'='*80}")
log(f"  {'variable':<14} {'unit':<6} {'rmse':>8} {'mae':>8} {'corr':>7} {'strict%':>9} {'fair%':>8} {'loose%':>8}")
for var, m in finalresults.items():
    unit = units.get(var, '?')
    log(f"  {var:<14} {unit:<6} {m['rmse']:>8.4f} {m['mae']:>8.4f} {m['corr']:>7.4f} "
          f"{m['acc05']:>8.2f}% {m['acc10']:>7.2f}% {m['acc20']:>7.2f}%")